<a href="https://colab.research.google.com/github/NatakiSystems/jupyterlab/blob/main/Nataki_Boykin_LLM_class_demos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# From Transformers to LLM Engineering — Hands-On Demos

You've just learned the concepts — now let's see them in action. This notebook has six short demos you can run yourself, tweak, and break on purpose. Each one connects directly to something from the lecture.

**Demos 1–5** call a real LLM through an API, so you'll need an API key. **Demo 6** runs entirely offline — no key or internet needed — so it's a good one to try first if you're still getting your key set up.

| # | Demo | What You'll See |
|---|------|------------------|
| 1 | Temperature Showdown | Why the same prompt gives different answers |
| 2 | The Persona Switch | How one instruction changes everything |
| 3 | Max Tokens Cliff | What happens when a response gets cut off |
| 4 | Zero-Shot vs. Few-Shot Race | Why examples make outputs more consistent |
| 5 | Break the JSON, Fix the JSON | Why real apps need error handling |
| 6 | Simulated Rate Limit + Backoff | Why retrying instantly is a bad idea |

Run the cells in order, read the notes before each one, and don't be afraid to change the prompts and re-run — that's the whole point.

## Setup

Run this once before starting Demos 1–5.

**Before you run it:** create a file named `.env` in the same folder as this notebook, and put your API key inside it like this:
```
DEEPSEEK_API_KEY=your_key_here
```
Never type your key directly into a code cell — that's the hardcoding mistake from the lecture, and it's how keys end up leaked.

In [ ]:
# Run this once to install what you need (skip if already installed)
# %pip install python-dotenv openai

In [ ]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

print("You're all set!" if os.getenv("DEEPSEEK_API_KEY") else "No API key found — check that your .env file exists and is spelled correctly.")

---
## Demo 1: Temperature Showdown

You just learned that temperature controls how "safe" vs. "exploratory" the model is when picking its next word. Let's prove it.

Below, the exact same prompt is sent six times — three times at `temperature=0`, then three times at `temperature=1`. Before you run it, guess: will the low-temperature answers look the same or different from each other? What about the high-temperature ones?

**Try it yourself after:** change the prompt to something else and see if the pattern holds.

In [ ]:
prompt = "Write one sentence describing a coffee maker."

for temp in [0, 0, 0, 1, 1, 1]:
    response = client.chat.completions.create(
        model="deepseek-chat",
        temperature=temp,
        messages=[{"role": "user", "content": prompt}]
    )
    print(f"[temp={temp}] {response.choices[0].message.content}\n")

**What did you notice?**

---
## Demo 2: The Persona Switch

Same question, three completely different system prompts. Before running each one, try to guess what the answer will sound like based on the persona alone.

**Try it yourself after:** write your own persona (e.g., "a Shakespearean actor" or "a no-nonsense drill sergeant") and swap it in.

In [ ]:
question = "What is a variable?"

personas = [
    "You are a pirate who explains everything in pirate speak.",
    "You are explaining this to a curious 5-year-old.",
    "You are a senior software engineer briefing a new hire."
]

for persona in personas:
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": persona},
            {"role": "user", "content": question}
        ]
    )
    print(f"--- Persona: {persona} ---")
    print(response.choices[0].message.content)
    print()

**What did you notice?** The question never changed — only the system prompt did. That one instruction reshaped tone, vocabulary, and even how technical the answer got. This is why role-setting goes in the system prompt: it should shape the *whole* conversation, not just one message.

---
## Demo 3: Max Tokens Cliff

`max_tokens` caps how long a response is allowed to be. Watch what happens when that cap is set way too low.

**Try it yourself after:** change 20 to something like 50 or 100 and see how the cutoff point moves.

In [ ]:
prompt = "Explain the history of the internet."

print("=== max_tokens = 20 ===")
response_short = client.chat.completions.create(
    model="deepseek-chat",
    max_tokens=20,
    messages=[{"role": "user", "content": prompt}]
)
print(response_short.choices[0].message.content)

print("\n=== max_tokens = 300 ===")
response_long = client.chat.completions.create(
    model="deepseek-chat",
    max_tokens=300,
    messages=[{"role": "user", "content": prompt}]
)
print(response_long.choices[0].message.content)

**What did you notice?** The first response probably stops mid-thought, maybe even mid-word. That's not a bug — the model got cut off because it hit its token limit. Remember: one token is roughly 4 characters, not a whole word, so it's easy to set this too low by accident.

---
## Demo 4: Zero-Shot vs. Few-Shot Race

We're going to invent a made-up task — turning casual text into "pirate professional" style — so the model has no existing pattern to fall back on. First we'll ask it with no examples (zero-shot). Then we'll show it 2 examples first (few-shot). Watch how much more consistent the format gets.

**Try it yourself after:** change `task_input` below to your own sentence and re-run both cells.

In [ ]:
task_input = "the meeting ran way over time"

zero_shot_prompt = f"Rewrite this as a 'pirate professional' announcement: {task_input}"

print("=== Zero-Shot (run 3x — watch the format/style vary) ===")
for _ in range(3):
    response = client.chat.completions.create(
        model="deepseek-chat",
        temperature=0.8,
        messages=[{"role": "user", "content": zero_shot_prompt}]
    )
    print(response.choices[0].message.content, "\n")

In [ ]:
few_shot_prompt = f"""Convert casual text into 'pirate professional' style, following the pattern below.

# Example 1
Casual: "hey can we push the meeting"
Pirate Professional: "Arr, might we be so bold as to reschedule our gathering, matey?"

# Example 2
Casual: "i need this by eod"
Pirate Professional: "Aye, this treasure be needed before the sun sets this very day."

# Now convert this:
Casual: "{task_input}"
Pirate Professional:"""

print("=== Few-Shot (run 3x — watch the format stay consistent) ===")
for _ in range(3):
    response = client.chat.completions.create(
        model="deepseek-chat",
        temperature=0.8,
        messages=[{"role": "user", "content": few_shot_prompt}]
    )
    print(response.choices[0].message.content, "\n")

**What did you notice?** The zero-shot answers probably varied in length, structure, and even how "pirate-y" they got. The few-shot answers should all follow roughly the same shape as the examples. Showing the model 2–3 examples of exactly what "good" looks like is one of the cheapest ways to get more reliable output — it just costs a few extra tokens.

---
## Demo 5: Break the JSON, Fix the JSON

Real apps don't want a friendly paragraph back from the model — they want structured data they can plug straight into code. Let's see what can go wrong, and how to guard against it.

**Step 1:** First, look at what happens when we try to parse JSON that's already broken (a trailing comma is invalid JSON).

In [ ]:
import json

broken_json = '{"name": "Alex", "skill_level": "beginner",}'  # trailing comma = invalid JSON

try:
    data = json.loads(broken_json)
except json.JSONDecodeError as e:
    print(f"JSONDecodeError caught: {e}")

**Step 2:** Now let's ask the model for JSON, but without telling it exactly what fields we want. Notice how loose and unpredictable the structure is.

In [ ]:
vague_prompt = "Give me info about a beginner Python student named Alex, in JSON."

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[{"role": "user", "content": vague_prompt}]
)
print(response.choices[0].message.content)

**Step 3:** Now we'll ask again, but this time we specify the exact fields we want and wrap the parsing in `try/except` — the same safety net a real application would use.

In [ ]:
strict_prompt = """Respond only with valid JSON. No preamble, no markdown formatting.
Include exactly these fields:
- name (string)
- skill_level (string)
- recommended_course (string)
- next_topic (string)

Student: Alex, a beginner interested in Python."""

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[{"role": "user", "content": strict_prompt}]
)

raw_output = response.choices[0].message.content

try:
    data = json.loads(raw_output)
    print("Parsed successfully:")
    print(data)
    print("\nRecommended course:", data["recommended_course"])
except json.JSONDecodeError:
    print("Model didn't return valid JSON — need a stricter prompt.")
    print("Raw output was:", raw_output)

**What did you notice?** Being specific about the schema made the output far more predictable and easy to use in code. But even with a good prompt, the `try/except` is still there — because the model can occasionally misbehave, and production code should never just assume the output will be perfect.

---
## Demo 6: Simulated Rate Limit + Backoff

**No API key or internet needed for this one** — it fakes a flaky API call so you can run it even if your key isn't working yet.

The function below is rigged to fail the first two times you call it and succeed on the third — just like a real API returning a `429 Too Many Requests` error. Watch how the code waits longer and longer between each retry (1s, 2s, 4s...) instead of hammering the server immediately.

**Try it yourself after:** change the `if attempt < 3` condition to `< 5` and see how the retry loop adapts.

In [ ]:
import time

attempt = 0

def flaky_call():
    """Simulates an API call that fails twice, then succeeds."""
    global attempt
    attempt += 1
    if attempt < 3:
        raise Exception("429 Too Many Requests")
    return "Success!"

for i in range(5):
    try:
        result = flaky_call()
        print(f"Attempt {i + 1}: {result}")
        break
    except Exception as e:
        wait = 2 ** i
        print(f"Attempt {i + 1} failed ({e}). Retrying in {wait}s...")
        time.sleep(wait)

**What did you notice?** Each retry waited longer than the last. If we retried instantly instead, we'd just be hammering an already-overloaded server, making things worse for everyone. This pattern — exponential backoff — is built into most provider SDKs by default, but it's worth knowing how it works under the hood.

---
## Wrap-Up

Look back at what you just ran: you controlled how creative the model is, how it talks, how long it talks, how consistent it is, how to handle its output safely, and how to deal with it when it fails. That's the whole arc — from "the model works" to "I can actually build something reliable with it."

**Before you go:** pick one demo above and change something — the prompt, the temperature, the persona, the schema — and see what happens. That kind of experimenting is exactly what prompt engineering looks like in the real world.